<a href="https://colab.research.google.com/github/tishasetia13/inside-out-emotion-classifier/blob/main/inside_out.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###GETTING OUR DATASET READY

In [ ]:
import pandas as pd

url = "https://huggingface.co/datasets/gsri-18/ISEAR-dataset-complete/resolve/main/ISEAR_dataset_complete.csv"
df = pd.read_csv(url)

In [ ]:
#df.head()
#df.shape
#df["emotion"].value_counts()
#df.isnull().sum()

In [ ]:
df = df[df['emotion'].isin(['joy', 'sadness', 'anger', 'fear', 'disgust'])]
df = df.drop(columns=['Unnamed: 2'])

In [ ]:
#df.shape
#df['emotion'].value_counts()

In [ ]:
df['content'] = df['content'].str.replace('\\n', ' ', regex=False)
df['content'] = df['content'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [ ]:
pd.set_option('display.max_colwidth', None)
df.sample(20, random_state=42)['content']
df['content'].str.len().describe()

,content
count,5395.000000
mean,112.438554
std,73.963873
min,6.000000
25%,58.000000
50%,96.000000
75%,150.000000
max,840.000000


In [ ]:
df = df.reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['emotion'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['emotion'], random_state=42)

print(train_df.shape, val_df.shape, test_df.shape)

(3776, 2) (809, 2) (810, 2)


###MAKING TOKENIZER AND DATA BATCHING

In [ ]:
from transformers import AutoTokenizer
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# sample = 'Hi, my name is Tisha'
# encoded = tokenizer(sample)
# print(encoded)
# print(tokenizer.convert_ids_to_tokens(encoded['input_ids']))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# lengths = [len(tokenizer.encode(text)) for text in train_df['content']]

# print(pd.Series(lengths).describe())
# print("95th percentile:", pd.Series(lengths).quantile(0.95))
# print("99th percentile:", pd.Series(lengths).quantile(0.99))

In [ ]:
# batch_texts = train_df['content'].tolist()[:4]

# batch = tokenizer(
#     batch_texts,
#     padding='max_length',
#     truncation=True,
#     max_length=64,
#     return_tensors='pt',
# )

# print(batch["input_ids"].shape)
# print(batch["attention_mask"].sum(dim=1))
# print(batch["input_ids"][0])

In [ ]:
labels = sorted(train_df['emotion'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

# print(label2id)
# print(id2label)

In [ ]:
for df in [train_df, val_df, test_df]:
  df['label'] = df['emotion'].map(label2id)

# print(train_df[["emotion", "label"]].head())
# print(train_df["label"].value_counts().sort_index())
# print("Missing labels:", train_df["label"].isna().sum(), val_df["label"].isna().sum(), test_df["label"].isna().sum())

In [ ]:
import torch

def encode_split(df):
  enc = tokenizer(
      df['content'].tolist(),
      padding='max_length',
      truncation=True,
      max_length=64,
      return_tensors='pt',
  )
  enc['labels'] = torch.tensor(df['label'].tolist())
  return enc

train_enc = encode_split(train_df)
val_enc = encode_split(val_df)
test_enc = encode_split(test_df)

# for name, enc in [("train", train_enc), ("val", val_enc), ("test", test_enc)]:
#     print(name, enc["input_ids"].shape, enc["attention_mask"].shape, enc["labels"].shape)
# print(train_enc["labels"][:5])

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_loader(enc, batch_size, shuffle):
  dataset = TensorDataset(enc['input_ids'], enc['attention_mask'], enc['labels'])
  return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_enc, batch_size=16, shuffle=True)
val_loader = make_loader(val_enc, batch_size=32, shuffle=False)
test_loader = make_loader(test_enc, batch_size=32, shuffle=False)

# print("Batches per epoch (train):", len(train_loader))
# ids, mask, y = next(iter(train_loader))
# print(ids.shape, mask.shape, y.shape)
# print(y)

### MODEL AND LEARNING

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# print("GPU available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU name:", torch.cuda.get_device_name(0))
# print("Using device:", device)
# needed = ["tokenizer", "label2id", "id2label", "train_enc", "val_enc",
#           "test_enc", "train_loader", "val_loader", "test_loader"]
# for name in needed:
#     print(f"{name:14s}", "OK" if name in globals() else "MISSING")

In [ ]:
from transformers import AutoModelForSequenceClassification

model_name='distilbert-base-uncased'

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
)

model = model.to(device)

# print('Model Class: ', type(model).__name__)
# print('Model is on: ', next(model.parameters()).device)

In [ ]:
# total_params = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Total parameters:     {total_params:,}")
# print(f"Trainable parameters: {trainable_params:,}")
# print()
# print(model)

In [ ]:
# batch = next(iter(train_loader))
# print("Type of batch:", type(batch))
# print("Number of items:", len(batch))
# for i, v in enumerate(batch):
#     print(f"item {i}: shape={tuple(v.shape)}  dtype={v.dtype}  device={v.device}")

In [ ]:
input_ids, attention_mask, labels = batch
# print("input_ids[0], first 15:     ", input_ids[0][:15].tolist())
# print("attention_mask[0], first 15:", attention_mask[0][:15].tolist())
# print("unique values in item 1:    ", torch.unique(attention_mask).tolist())
# print("labels:", labels.tolist())
# print()
# print("Decoded example 0:")
# print(tokenizer.decode(input_ids[0], skip_special_tokens=False))

In [ ]:
# input_ids = input_ids.to(device)
# attention_mask = attention_mask.to(device)
# labels = labels.to(device)

# model.eval()
# with torch.no_grad():
#   outputs = model(input_ids=input_ids,
#                       attention_mask=attention_mask,
#                       labels=labels)

# print("logits shape:", tuple(outputs.logits.shape))
# print("loss:", outputs.loss.item())
# print("expected loss for a clueless model (ln 5): 1.609")
# print()
# print("logits for example 0:", outputs.logits[0].tolist())
# print("predicted class ids: ", outputs.logits.argmax(dim=-1).tolist())
# print("true labels:         ", labels.tolist())

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay: ", optimizer.param_groups[0]["weight_decay"])

In [ ]:
# model.eval()
# with torch.no_grad():
#   loss_before = model(input_ids=input_ids,
#                           attention_mask=attention_mask,
#                           labels=labels).loss.item()
# w_before = model.classifier.weight.detach().clone()

# model.train()
# optimizer.zero_grad()
# outputs = model(input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 labels=labels)
# loss = outputs.loss
# loss.backward()
# grad_size = model.classifier.weight.grad.norm().item()
# optimizer.step()

# model.eval()
# with torch.no_grad():
#     loss_after = model(input_ids=input_ids,
#                        attention_mask=attention_mask,
#                        labels=labels).loss.item()

# change = (model.classifier.weight.detach() - w_before).abs().max().item()

# print(f"loss before step:            {loss_before:.4f}")
# print(f"loss during step (train):    {loss.item():.4f}")
# print(f"loss after step:             {loss_after:.4f}")
# print(f"gradient size (classifier):  {grad_size:.4f}")
# print(f"biggest weight change:       {change:.8f}")

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0

  for batch in train_loader:
    input_ids, attention_mask, labels = [t.to(device) for t in batch]
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
    loss = outputs.loss
    loss.backward()
    torch.nn.utils.clip_grad_norm(model.parameters(), 1.0)
    optimizer.step()
    running_loss += loss.item()

  avg_loss = running_loss / len(train_loader)
  print(f"Epoch {epoch + 1}/{num_epochs}  train loss: {avg_loss:.4f}")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/inside_out_model"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved files:", os.listdir(save_dir))

In [ ]:
model.eval()
val_loss_total = 0.0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids, attention_mask, labels = [t.to(device) for t in batch]

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        val_loss_total += outputs.loss.item()
        preds = outputs.logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

val_loss = val_loss_total / len(val_loader)
correct = sum(p == l for p, l in zip(all_preds, all_labels))
val_acc = correct / len(all_labels)

print(f"Validation loss:     {val_loss:.4f}")
print(f"Validation accuracy: {val_acc:.4f}  ({correct}/{len(all_labels)})")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

target_names = [id2label[i] for i in range(5)]

print(classification_report(all_labels, all_preds, target_names=target_names, digits=3))
print("Confusion matrix (rows = true, columns = predicted):")
print(target_names)
print(confusion_matrix(all_labels, all_preds))

In [ ]:
model.eval()
test_loss_total = 0.0
test_preds = []
test_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [t.to(device) for t in batch]

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)

        test_loss_total += outputs.loss.item()
        preds = outputs.logits.argmax(dim=-1)
        test_preds.extend(preds.cpu().tolist())
        test_labels.extend(labels.cpu().tolist())

test_loss = test_loss_total / len(test_loader)
correct = sum(p == l for p, l in zip(test_preds, test_labels))
test_acc = correct / len(test_labels)

print(f"Test loss:     {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}  ({correct}/{len(test_labels)})")

In [ ]:
# INFERENCE

import torch.nn.functional as F

def predict_emotion(text):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)

    with torch.no_grad():
        logits = model(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"]).logits

    probs = F.softmax(logits, dim=-1)[0]
    ranked = sorted(zip(target_names, probs.tolist()), key=lambda x: -x[1])

    print(f"Text: {text}")
    for name, p in ranked:
        print(f"   {name:8s} {p:.1%}")
    print()

predict_emotion("I finally got the internship offer I was dreaming of!")
predict_emotion("I felt sick seeing the food left out in the heat for days.")
predict_emotion("My best friend lied to me and I couldn't stop shaking with rage.")

Text: I finally got the internship offer I was dreaming of!
   joy      99.0%
   fear     0.4%
   sadness  0.3%
   disgust  0.2%
   anger    0.1%

Text: I felt sick seeing the food left out in the heat for days.
   disgust  92.2%
   sadness  4.4%
   anger    1.4%
   fear     1.0%
   joy      0.9%

Text: My best friend lied to me and I couldn't stop shaking with rage.
   anger    73.9%
   sadness  10.8%
   disgust  9.3%
   fear     5.6%
   joy      0.6%



In [3]:
## MAIN

import os
import torch
from google.colab import drive
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/inside_out_model"
model = AutoModelForSequenceClassification.from_pretrained(save_dir).to(device)
model.eval()

target_names = [model.config.id2label[i] for i in range(5)]

print("Device:", device)
print("Model on:", next(model.parameters()).device)
print("Labels:", target_names)

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Device: cuda
Model on: cuda:0
Labels: ['anger', 'disgust', 'fear', 'joy', 'sadness']


In [ ]:
# INFERENCE TESTING

easy = [
    "I got the highest marks in my class and my parents hugged me.",           # joy
    "I was walking alone at night and heard footsteps right behind me.",       # fear
    "My roommate ate my food again after I told her not to.",                  # anger
    "I found a dead rat in the kitchen cupboard.",                             # disgust
    "I cried all night when my grandfather passed away.",                      # sadness
]

realistic = [
    "The airline lost my bag and then the staff just shrugged at me.",         # anger
    "I have my viva tomorrow and I can't stop my hands from trembling.",       # fear
    "My best friend moved to another country and the house feels so empty.",   # sadness
    "Someone sneezed straight into the food at the buffet and kept serving.",  # disgust
    "My little brother took his first steps today and I got it on video!",     # joy
]

tricky = [
    "I thought I would be happy about the promotion, but I just feel numb.",   # sadness? joy?
    "He smiled at me while telling everyone my secret.",                       # anger? disgust?
    "I wasn't scared at all.",                                                 # negation
    "Oh great, another power cut right before my exam.",                       # sarcasm
    "I laughed so hard at the funeral, and then I felt awful about it.",       # mixed
]

casual = [
    "omg just got selected for the hackathon finals!!!",                       # joy (texting style)
    "ugh my landlord kept my deposit for no reason",                           # anger
    "bro the washroom at the station was soooo gross",                         # disgust
]

for group in [easy, realistic, tricky, casual]:
    for s in group:
        predict_emotion(s)
    print("=" * 40)

Text: I got the highest marks in my class and my parents hugged me.
   joy      98.9%
   sadness  0.5%
   fear     0.3%
   disgust  0.2%
   anger    0.2%

Text: I was walking alone at night and heard footsteps right behind me.
   fear     98.9%
   joy      0.4%
   sadness  0.3%
   disgust  0.3%
   anger    0.2%

Text: My roommate ate my food again after I told her not to.
   disgust  53.2%
   anger    42.1%
   sadness  3.3%
   joy      0.9%
   fear     0.5%

Text: I found a dead rat in the kitchen cupboard.
   disgust  96.0%
   fear     1.7%
   sadness  1.0%
   anger    0.9%
   joy      0.4%

Text: I cried all night when my grandfather passed away.
   sadness  98.2%
   joy      0.6%
   fear     0.6%
   anger    0.3%
   disgust  0.3%

Text: The airline lost my bag and then the staff just shrugged at me.
   anger    82.2%
   disgust  11.3%
   sadness  4.8%
   joy      1.0%
   fear     0.6%

Text: I have my viva tomorrow and I can't stop my hands from trembling.
   joy      62.4%
   sadne

In [ ]:
# INFERENCE TESTING

def predict_story(text, top_k=3):
    model.eval()

    n_tokens = len(tokenizer(text)["input_ids"])
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).to(device)

    with torch.no_grad():
        logits = model(input_ids=enc["input_ids"],
                       attention_mask=enc["attention_mask"]).logits

    probs = F.softmax(logits, dim=-1)[0]
    ranked = sorted(zip(target_names, probs.tolist()), key=lambda x: -x[1])

    print(f"Tokens: {n_tokens}", end="   ")
    if n_tokens > 64:
        print(f"-> TRUNCATED, model missed the last {n_tokens - 64} tokens")
        seen = tokenizer.decode(enc["input_ids"][0], skip_special_tokens=True)
        print("   Model saw until: ...", seen[-60:])
    else:
        print("-> fits, model saw everything")

    for name, p in ranked[:top_k]:
        print(f"   {name:8s} {p:.1%}")
    print()


stories = [
    # joy, fits
    "For months I had been practicing for the state level debate. When the judges announced my name as the winner, I could not stop smiling. My whole family was cheering in the audience and my mother was crying happily.",

    # fear, fits
    "Last summer I was trekking with my cousins when the weather suddenly changed. The fog became so thick that we lost the trail. When I heard a growl somewhere close in the dark, my heart started pounding and I froze.",

    # anger, emotion at the START of a long story
    "I was furious when I found out my group partner had not done any of her work. I had stayed up for three nights writing the report, rewriting her sections and fixing the slides. On the day of the presentation she walked in late, stood in front of the teacher and took credit for everything while I stood there silently.",

    # sadness, emotion only at the END of a long story
    "It was a regular Sunday. I woke up late, made tea and cleaned my room. In the afternoon we went to the market to buy vegetables and fruits for the week, and my dad stopped to chat with a shopkeeper he had known for years. When we came back home, my aunt called. My grandmother, who had raised me, had passed away in her sleep, and I sat on the floor unable to speak.",

    # disgust, long
    "We went to a famous restaurant for my sister's birthday dinner. The table was sticky and the menu smelled of old oil. Halfway through the meal I saw a cockroach crawling out of the plate of rice that was meant for our table, and the waiter just wiped it away with his bare hand and walked off.",

    # mixed, starts scary, ends happy
    "I was so afraid walking into the hall for my interview that my knees were shaking and I forgot my own name for a second. But the panel was kind, the questions went well, and an hour later they called to say I was selected. I ran outside and laughed with relief.",
]

for s in stories:
    print("STORY:", s[:70] + "...")
    predict_story(s)
    print("-" * 50)

STORY: For months I had been practicing for the state level debate. When the ...
Tokens: 45   -> fits, model saw everything
   joy      97.6%
   sadness  1.6%
   fear     0.3%

--------------------------------------------------
STORY: Last summer I was trekking with my cousins when the weather suddenly c...
Tokens: 47   -> fits, model saw everything
   fear     98.5%
   joy      0.5%
   sadness  0.4%

--------------------------------------------------
STORY: I was furious when I found out my group partner had not done any of he...
Tokens: 68   -> TRUNCATED, model missed the last 4 tokens
   Model saw until: ...  front of the teacher and took credit for everything while i
   anger    95.9%
   disgust  2.0%
   sadness  1.0%

--------------------------------------------------
STORY: It was a regular Sunday. I woke up late, made tea and cleaned my room....
Tokens: 88   -> TRUNCATED, model missed the last 24 tokens
   Model saw until: ...  known for years. when we came back home, my aunt ca

In [4]:
!ls /content/drive/MyDrive/inside_out_model

config.json  model.safetensors	tokenizer_config.json  tokenizer.json


In [5]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

In [6]:
from huggingface_hub import whoami

print(whoami()["name"])

tishasetia13


In [7]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "tishasetia13/inside-out-emotion-classifier"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path="/content/drive/MyDrive/inside_out_model",
    repo_id=repo_id,
    repo_type="model",
)

CommitInfo(commit_url='https://huggingface.co/tishasetia13/inside-out-emotion-classifier/commit/80070baf87f3a940443da925d1c356a0bd018c28', commit_message='Upload folder using huggingface_hub', commit_description='', oid='80070baf87f3a940443da925d1c356a0bd018c28', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tishasetia13/inside-out-emotion-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='tishasetia13/inside-out-emotion-classifier'), pr_revision=None, pr_num=None)

In [8]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

repo_id = "tishasetia13/inside-out-emotion-classifier"

tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForSequenceClassification.from_pretrained(repo_id)

print(model.config.id2label)

config.json:   0%|          | 0.00/854 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{0: 'anger', 1: 'disgust', 2: 'fear', 3: 'joy', 4: 'sadness'}
